# M08. Stop the world

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/m08-stop-the-world/m08.ipynb)

M07 spent a whole lesson on three lists. An object starts in the first one, gets promoted every time it survives a pass, and never comes back down, which is why a cycle that has been alive for a while cannot be freed by a pass over the youngest list.

On the [free threaded build](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#free-threaded-build) those three lists do not exist.

![the ordinary build's three linked lists against the free threaded build's single heap](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m08-stop-the-world/diagrams/two-shapes-of-heap.svg)

This lesson runs M07's experiments again on that build. The `gc` module still answers all the same questions, because the language documents them, but several of the answers stop meaning what they used to. Then it gets to the part that has no equivalent on the ordinary build at all, which is what happens to the other threads while a collection is running.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Python/gc_free_threading.c:1995-2015@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Every cell runs on the interpreter you already have, including in a browser. There is nothing to build and nothing to install beyond the cell above.

Three of the things this lesson is about only exist in a build made without the GIL, and you almost certainly are not running one. Those three arrive as recordings: the program, and what it printed when it ran in the image this project publishes. You can read the program, run the parts of it that work on your own build, and pull the same image if you want to see it for yourself.

None of the cells here start threads, so all of them work in a browser too.

## Which interpreter is this

In [ ]:
import pyxray

pyxray.show()

## Which build you are on

Two ways to ask, and they answer slightly different questions. `sys._is_gil_enabled()` tells you whether the GIL is switched on right now. The build setting tells you whether this interpreter was compiled with the option at all.

An ordinary CPython build reports that the GIL is enabled and has no Py_GIL_DISABLED setting, and a free threaded build reports the opposite

In [ ]:
import sysconfig

print(f"  sys._is_gil_enabled()          {sys._is_gil_enabled()}")
print(f"  Py_GIL_DISABLED build setting  {sysconfig.get_config_var('Py_GIL_DISABLED')}")

if sysconfig.get_config_var("Py_GIL_DISABLED"):
    print("  you are on a free threaded build, so the cells below will match the recordings")
else:
    print("  you are on an ordinary build, so the cells below show the other half of each")
    print("  comparison and the recordings show what the free threaded build does")

## Three lists, or none

`struct _gc_runtime_state` is where the collector keeps its state, and the shape of it depends on which build you compiled. On an ordinary build it holds an array of three `gc_generation` structures, each with a threshold, a count, and the head of a linked list that objects get chained onto. On a free threaded build it holds a `young` and two `old` structures with the thresholds and counts, and no lists [Include/internal/pycore_interp_structs.h:228-234@v3.15.0rc1#generations](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_interp_structs.h#L228-L234).

The thresholds are the same numbers on both, 2000 and 10 and 10 [Include/internal/pycore_interp_structs.h:279-286@v3.15.0rc1#GC_GENERATION_INIT](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_interp_structs.h#L279-L286). They are just counting something with nothing underneath it.

So where does the collector find your object? It asks the memory allocator. Every thread on this build gets its own [mimalloc heap](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#mimalloc-heap), and one of those heaps is reserved for objects the collector cares about, so a pass means walking the allocator's own pages rather than following a chain the interpreter maintained [Python/gc_free_threading.c:368-395@v3.15.0rc1#gc_visit_heaps_lock_held](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L368-L395).

That removes a pointer pair from every tracked object, which is the sixteen bytes M06 measured. What was in the list is now a bit in the object header instead [Include/internal/pycore_gc.h:39-45@v3.15.0rc1#_PyGC_BITS_TRACKED](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_gc.h#L39-L45).

![the seven collector bits packed into one byte of the free threaded object header](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m08-stop-the-world/diagrams/the-bits-that-replaced-the-lists.svg)

M06 found that byte by reading memory: `ob_gc_bits` sits at offset 11, right after `ob_mutex`. Seven of its eight bits are spoken for, and four of them are things the collector writes during a pass.

## The same three cells, different answers

Here is M07's promotion cell again, unchanged. On an ordinary build it prints 0, then 1, then 2.

gc.get_objects takes a generation, and on an ordinary build the three generations hold different numbers of objects and an ordinary dictionary is in exactly one of them

In [ ]:
import gc

gc.disable()
gc.collect()
mine = {"tag": "follow me"}


def generations_holding(obj):
    return [g for g in range(3) if any(o is obj for o in gc.get_objects(generation=g))]


print(f"  objects in each generation  {[len(gc.get_objects(generation=g)) for g in range(3)]}")
print(f"  as soon as it exists        generations {generations_holding(mine)}")
gc.collect(0)
print(f"  after a pass over gen 0     generations {generations_holding(mine)}")
gc.collect(1)
print(f"  after a pass over gen 1     generations {generations_holding(mine)}")
gc.enable()

> **Version note.** How many objects your session has is your session's business. What matters is that the three numbers are different from each other and that the dictionary is in one generation at a time. On the build Pyodide ships the middle line reads 2 rather than 1, for the reason M07 described.

And here is M07's ageing cell, also unchanged. On an ordinary build the old cycle survives, because it is not in the list that pass walks.

On an ordinary build a cycle made a moment ago is freed by gc.collect(0) and an identical cycle that has survived five passes is not

In [ ]:
import weakref


class Node:
    def __init__(self):
        self.peer = None


def make_cycle():
    left, right = Node(), Node()
    left.peer = right
    right.peer = left
    return left


gc.collect()
fresh = make_cycle()
watch_fresh = weakref.ref(fresh)
del fresh
gc.collect(0)
print(f"  a cycle made a moment ago, freed by gc.collect(0)  {watch_fresh() is None}")

gc.collect()
older = make_cycle()
watch_older = weakref.ref(older)
for _ in range(5):
    gc.collect(0)
del older
gc.collect(0)
print(f"  a cycle that survived five passes, same call       {watch_older() is None}")

Now the same two cells on the other build.

On a free threaded build all three generations report the same objects, a new dictionary is in all three at once, and gc.collect(0) frees a cycle that has survived five passes

Which generation is an object in on a build that does not keep generation lists?

```python
"""Ask this build which generation an object is in, and get three answers.

M07 established three things about the ordinary build. An object starts in generation 0 and gets
promoted every time it survives a pass. `gc.get_objects` takes a generation and shows you which
list an object is in. And a cycle that has already survived a few passes cannot be freed by
`gc.collect(0)`, because it is not in the list that pass walks.

None of that is true here. This build has no generation lists at all. `struct _gc_runtime_state`
keeps a `young` counter and two `old` counters and nothing to hang objects off, because the
collector walks the memory allocator's heaps rather than a linked list it maintains itself. So
every collection walks everything, and the generation number you pass in only decides which
counters get reset afterwards.
"""

import gc
import sys
import sysconfig
import weakref

print(f"    python {sys.version.split()[0]}")
print(f"    gil enabled: {sys._is_gil_enabled()}")
print(f"    Py_GIL_DISABLED: {sysconfig.get_config_var('Py_GIL_DISABLED')}")
print()

gc.disable()
gc.collect()

print("    gc.get_objects takes a generation. Ask it for each of the three:")
sizes = [len(gc.get_objects(generation=g)) for g in range(3)]
for generation, size in enumerate(sizes):
    print(f"      generation {generation}: {size} objects")
print(f"~ the three generations hold the same objects: {sizes[0] == sizes[1] == sizes[2]}")
print()

mine = {"tag": "follow me"}


def generations_holding(obj):
    return [g for g in range(3) if any(o is obj for o in gc.get_objects(generation=g))]


print("    Follow one ordinary dictionary through the passes that promoted it in M07:")
print(f"      as soon as it exists      generations {generations_holding(mine)}")
gc.collect(0)
print(f"      after a pass over gen 0   generations {generations_holding(mine)}")
gc.collect(1)
print(f"      after a pass over gen 1   generations {generations_holding(mine)}")
gc.collect(2)
print(f"      after a full pass         generations {generations_holding(mine)}")
print(f"~ generations the dictionary is reported in after every pass: {generations_holding(mine)}")
print()


class Node:
    """A cycle of two of these is unreachable garbage that only the collector can free."""

    def __init__(self, tag):
        self.tag = tag
        self.other = None


def make_cycle(tag):
    left = Node(tag)
    right = Node(tag)
    left.other = right
    right.other = left
    return left


gc.collect()
fresh = make_cycle("fresh")
watch_fresh = weakref.ref(fresh)
del fresh
gc.collect(0)
print(f"    a cycle made a moment ago, freed by gc.collect(0): {watch_fresh() is None}")

gc.collect()
older = make_cycle("older")
watch_older = weakref.ref(older)
for _ in range(5):
    gc.collect(0)
del older
gc.collect(0)
print(f"    a cycle that survived five passes, same call:      {watch_older() is None}")
print("~ a pass over generation 0 frees an old cycle on this build: True")
print()

print("    The thresholds are still three numbers, and gc.get_count still returns three,")
print("    because the module has to keep answering the questions the language documents.")
print(f"      gc.get_threshold()  {gc.get_threshold()}")
print(f"      gc.get_count()      {gc.get_count()}")
print("    But the second and third are counts of collections, not lists of objects, and")
print("    there is nothing underneath them to walk separately.")
gc.enable()
```

```text
    python 3.15.0rc1
    gil enabled: False
    Py_GIL_DISABLED: 1

    gc.get_objects takes a generation. Ask it for each of the three:
      generation 0: 7116 objects
      generation 1: 7116 objects
      generation 2: 7116 objects
~ the three generations hold the same objects: True

    Follow one ordinary dictionary through the passes that promoted it in M07:
      as soon as it exists      generations [0, 1, 2]
      after a pass over gen 0   generations [0, 1, 2]
      after a pass over gen 1   generations [0, 1, 2]
      after a full pass         generations [0, 1, 2]
~ generations the dictionary is reported in after every pass: [0, 1, 2]

    a cycle made a moment ago, freed by gc.collect(0): True
    a cycle that survived five passes, same call:      True
~ a pass over generation 0 frees an old cycle on this build: True

    The thresholds are still three numbers, and gc.get_count still returns three,
    because the module has to keep answering the questions the language documents.
      gc.get_threshold()  (2000, 10, 10)
      gc.get_count()      (312, 6, 0)
    But the second and third are counts of collections, not lists of objects, and
    there is nothing underneath them to walk separately.
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

The generation argument is still accepted. `gc.get_objects` just hands it to a function that ignores it and walks the whole heap, stopping every other thread to do so [Python/gc_free_threading.c:2435-2451@v3.15.0rc1#_PyGC_GetObjects](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2435-L2451). And `gc.collect(0)` runs the same collection `gc.collect(2)` would; the generation number only decides which counters get reset afterwards [Python/gc_free_threading.c:2064-2078@v3.15.0rc1#gc_collect_internal](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2064-L2078).

![five questions from M07 with the answer each build gives](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m08-stop-the-world/diagrams/same-question-two-answers.svg)

Which is a real trade. There is no cheap young pass on this build, so a program making a lot of short lived garbage cannot get away with a quick look at the recent stuff. Every pass is the expensive one. In exchange, no cycle is ever invisible because of its age, and nothing has to maintain a linked list under concurrent allocation.

## The count nobody owns

M07 established that the counter is tracked objects alive right now, and that it is exact. On this build the first part holds and the second does not.

The problem is that an atomic add on one shared word, for every tracked object any thread makes, would put a contention point on one of the hottest paths in the interpreter. So each thread keeps a private running total and only pushes it into the shared count once it has built up 512 [Python/gc_free_threading.c:44-46@v3.15.0rc1#LOCAL_ALLOC_COUNT_THRESHOLD](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L44-L46) [Python/gc_free_threading.c:2017-2037@v3.15.0rc1#record_allocation](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2017-L2037). Deallocations go the same way, with the sign flipped [Python/gc_free_threading.c:2039-2062@v3.15.0rc1#record_deallocation](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2039-L2062).

![an allocation raising a thread local counter that only reaches the shared count every 512 objects](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m08-stop-the-world/diagrams/the-count-nobody-owns.svg)

Reading it from the thread that did the allocating hides all of this, because `gc.get_count` flushes the calling thread's own buffer before it answers [Modules/gcmodule.c:215-240@v3.15.0rc1#gc_get_count_impl](https://github.com/python/cpython/blob/v3.15.0rc1/Modules/gcmodule.c#L215-L240). So the exactness you see below is real on both builds.

Making a number of tracked objects and immediately reading gc.get_count from the same thread moves the first number by exactly that many

In [ ]:
gc.disable()
gc.collect()
base = gc.get_count()[0]
kept = []

print("  objects made    change gc.get_count reports")
for target in (200, 400, 600, 800, 1000, 1200):
    while len(kept) < target:
        kept.append([])
    print(f"  {target:>12}    {gc.get_count()[0] - base:>26}")
del kept
gc.enable()

> **Version note.** The right hand column should match the left one almost exactly, give or take a handful of objects the loop itself made. Reading from the allocating thread is exact on every build, which is the point of the recording below.

Now read it from a thread that is not the one allocating.

On a free threaded build a helper thread that has made 400 tracked objects moves the count another thread reads by about 5, and the number the reader sees only moves in steps of 512

How stale is the collector's count when another thread is the one doing the allocating?

```python
"""The collector's counter, read from a thread that did not do the allocating.

On the ordinary build there is one counter and one thread touching it at a time, so it is exact.
Here every thread can allocate at once, and an atomic add on the same word for every object any
thread makes would put a contention point on one of the hottest paths in the interpreter.

So each thread keeps its own running total and only pushes it into the shared count once it has
built up 512 of them. That makes the shared number cheap and approximate. This program measures
how approximate, by having one thread allocate while another reads.
"""

import gc
import sys
import sysconfig
import threading

print(f"    python {sys.version.split()[0]}")
print(f"    gil enabled: {sys._is_gil_enabled()}")
print(f"    Py_GIL_DISABLED: {sysconfig.get_config_var('Py_GIL_DISABLED')}")
print()

gc.disable()
gc.collect()

#: How many objects the helper makes before handing back to the main thread to read the count.
STEP = 200
#: How many times it does that.
ROUNDS = 8

made = threading.Event()
carry_on = threading.Event()
kept = []


def helper():
    """Make STEP tracked objects, hand back to the main thread, repeat."""
    for _ in range(ROUNDS):
        for _ in range(STEP):
            kept.append([])
        made.set()
        carry_on.wait()
        carry_on.clear()


worker = threading.Thread(target=helper)
base = gc.get_count()[0]
worker.start()

print("    objects the helper made    change the main thread can see")
seen = 0
for round_number in range(1, ROUNDS + 1):
    made.wait()
    made.clear()
    seen = gc.get_count()[0] - base
    print(f"      {round_number * STEP:>22}    {seen:>27}")
    carry_on.set()

print()
print(f"~ objects one thread made while another watched: {STEP * ROUNDS}")
print(f"~ change the watching thread could see: {seen}")
print()
print("    The number the main thread reads moves in jumps of 512, which is the constant")
print("    LOCAL_ALLOC_COUNT_THRESHOLD, and it only moves when the helper crosses a multiple")
print("    of it. Between those points the main thread is reading a count that is behind by")
print("    up to 512 for every other thread that is running.")
print()
print("    Reading it from the thread that did the allocating is exact, because gc.get_count")
print("    flushes the calling thread's own buffer before it answers. Only the other threads")
print("    are stale, and only until they fill their buffer.")
print()
worker.join()
print(f"    objects the helper actually made: {len(kept)}")
print(f"    what the main thread sees now the helper has exited: {gc.get_count()[0] - base}")
print("    A thread flushes what is left in its buffer on the way out, which is why that")
print("    last number is exact and every number above it was not.")
gc.enable()
```

```text
    python 3.15.0rc1
    gil enabled: False
    Py_GIL_DISABLED: 1

    objects the helper made    change the main thread can see
                         200                              5
                         400                              5
                         600                            517
                         800                            517
                        1000                            517
                        1200                           1029
                        1400                           1029
                        1600                           1541

~ objects one thread made while another watched: 1600
~ change the watching thread could see: 1541

    The number the main thread reads moves in jumps of 512, which is the constant
    LOCAL_ALLOC_COUNT_THRESHOLD, and it only moves when the helper crosses a multiple
    of it. Between those points the main thread is reading a count that is behind by
    up to 512 for every other thread that is running.

    Reading it from the thread that did the allocating is exact, because gc.get_count
    flushes the calling thread's own buffer before it answers. Only the other threads
    are stale, and only until they fill their buffer.

    objects the helper actually made: 1600
    what the main thread sees now the helper has exited: 1605
    A thread flushes what is left in its buffer on the way out, which is why that
    last number is exact and every number above it was not.
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

The number the watching thread sees goes 5, 5, 517, 517, 517, 1029, and so on. It is not wrong so much as behind, by up to 512 for every other thread that is running.

Which is fine, because of what the number is for. Nothing reads it to make a decision that has to be correct. It is read to decide whether to schedule a collection, and being a few hundred objects late on that is not a problem [Python/gc_free_threading.c:1995-2015@v3.15.0rc1#gc_should_collect](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L1995-L2015).

That function is worth a second look, because the brake M07 spent a section on has moved. On the ordinary build the quarter rule only guards the full pass. Here there is only one kind of pass, so the same rule guards all of them: a collection is skipped unless the young count has reached a quarter of the objects that survived the last one.

## Stop the world

Here is the part with no equivalent on the ordinary build.

The collector has to work out which objects are only kept alive by each other. Doing that means comparing every object's reference count against the number of references it can find, which only works if nothing is taking or dropping a reference while it looks. On the ordinary build the GIL supplies that for free. Here it has to be arranged.

So the collection starts by stopping every other thread [Python/gc_free_threading.c:2064-2078@v3.15.0rc1#gc_collect_internal](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2064-L2078), and does not start them again until it has found the garbage [Python/gc_free_threading.c:2141-2161@v3.15.0rc1#_PyEval_StartTheWorld](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2141-L2161). Finalizers and weakref callbacks run afterwards, with the world going again, because those are arbitrary Python code and running them with everything parked is a good way to deadlock.

[stop the world](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#stop-the-world) sounds like a special mechanism and it is not. A thread that is running Python gets a bit set on its eval breaker, the same word M07 watched the collector get scheduled through, and it parks itself between two bytecode instructions [Include/internal/pycore_ceval.h:348-353@v3.15.0rc1#_PY_EVAL_PLEASE_STOP_BIT](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_ceval.h#L348-L353). A thread that is already blocked in C, waiting on a socket or a lock, is marked parked where it stands and never woken at all [Python/pystate.c:2385-2406@v3.15.0rc1#park_detached_threads](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2385-L2406). The collector then waits in one millisecond steps until the last one has stopped [Python/pystate.c:2408-2443@v3.15.0rc1#stop_the_world](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2408-L2443).

![how each kind of thread gets stopped and what stopping it costs](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m08-stop-the-world/diagrams/what-stopping-costs.svg)

How long they stay stopped is how long the pass takes, and you can measure that on any build.

A full pass over a heap with two hundred thousand cycles on it takes a measurable number of milliseconds, and a pass over the same heap once those cycles are gone takes almost none

In [ ]:
import time

gc.disable()
gc.collect()

heap = []
for _ in range(200000):
    left, right = Node(), Node()
    left.peer = right
    right.peer = left
    heap.append(left)

started = time.perf_counter()
gc.collect()
took = time.perf_counter() - started
print(f"  cycles on the heap             {len(heap)}")
print(f"  one full pass over it          {took * 1000:.0f} ms")

del heap
gc.collect()
started = time.perf_counter()
gc.collect()
print(f"  and over an empty one          {(time.perf_counter() - started) * 1000:.0f} ms")
gc.enable()

> **Version note.** The milliseconds depend entirely on your machine, and in a browser they are roughly double. The shape is what matters: the first number is a real amount of time and the second is close to nothing, because the cost of a pass is the size of the heap.

That number is a pause on a free threaded build. Every other thread in the process is parked for it, whatever it was doing.

Three threads spinning in a loop that touches nothing lose almost exactly three times the time the collector spends, which is what being stopped for the whole pass looks like

How much time does a thread with no interest in the collector lose to a collection?

```python
"""What the other threads are doing while the collector walks the heap.

The free threaded build removed the GIL, so several threads really do run Python at the same
time. It did not remove the collector's need to look at a heap that nothing is modifying. So
before a collection starts, every other thread is stopped, and it stays stopped until the
collector has found the garbage.

This program puts three threads in a tight loop that does nothing but read the clock and add up
how long it spent not running. Then it runs the same loop again with collections happening
underneath it. The difference between the two totals is time that threads with no interest in
the collector lost to it.
"""

import gc
import sys
import sysconfig
import threading
import time

print(f"    python {sys.version.split()[0]}")
print(f"    gil enabled: {sys._is_gil_enabled()}")
print(f"    Py_GIL_DISABLED: {sysconfig.get_config_var('Py_GIL_DISABLED')}")
print()

#: How many two node cycles to leave on the heap for the collector to walk.
CYCLES = 300000
#: How many threads spin in the measuring loop.
WORKERS = 3
#: How many collections to run during the second measurement.
PASSES = 5
#: A gap longer than this counts as the thread having been stopped rather than merely descheduled.
STALL = 0.001


class Node:
    """Two of these pointing at each other is a cycle only the collector can free."""

    __slots__ = ("peer",)

    def __init__(self):
        self.peer = None


heap = []
for _ in range(CYCLES):
    left, right = Node(), Node()
    left.peer = right
    right.peer = left
    heap.append(left)

gc.disable()
gc.collect()


def measure(collections):
    """Spin WORKERS threads for a moment. Return their total stalled time and how long
    the collections themselves took."""
    stalled = [0.0] * WORKERS
    running = True

    def busy(slot):
        lost = 0.0
        last = time.perf_counter()
        while running:
            now = time.perf_counter()
            if now - last > STALL:
                lost += now - last
            last = now
        stalled[slot] = lost

    threads = [threading.Thread(target=busy, args=(number,)) for number in range(WORKERS)]
    for thread in threads:
        thread.start()
    time.sleep(0.2)
    started = time.perf_counter()
    for _ in range(collections):
        gc.collect()
    collecting = time.perf_counter() - started
    time.sleep(0.2)
    running = False
    for thread in threads:
        thread.join()
    return sum(stalled), collecting


measure(0)
quiet, _ = measure(0)
loud, collecting = measure(PASSES)

print(f"    {CYCLES} cycles on the heap, {WORKERS} threads spinning, nothing shared between them")
print()
print(f"~ seconds the collector spent on {PASSES} passes: {collecting:.3f}")
print(f"~ seconds the three threads lost with nothing collecting: {quiet:.3f}")
print(f"~ seconds the three threads lost with those passes running: {loud:.3f}")
per_pass = (loud - quiet) / WORKERS / PASSES * 1000
print(f"~ lost per thread per pass, in milliseconds: {per_pass:.0f}")
print()
print("    Those threads never touched the heap the collector was walking and never called")
print("    anything in the gc module. They were stopped anyway, because the collector needs")
print("    every reference count in the process to hold still while it works out which ones")
print("    are only kept alive by the cycle it is looking at.")
print()
print("    The stopping uses the same machinery M07 described. A thread that is running Python")
print("    gets a bit set on its eval breaker and parks itself between two bytecode")
print("    instructions. A thread that is already blocked in C, waiting on a socket or a lock,")
print("    is marked parked without being woken at all, which is why a program full of threads")
print("    waiting on IO costs the collector nothing to stop.")
gc.enable()
```

```text
    python 3.15.0rc1
    gil enabled: False
    Py_GIL_DISABLED: 1

    300000 cycles on the heap, 3 threads spinning, nothing shared between them

~ seconds the collector spent on 5 passes: 0.042
~ seconds the three threads lost with nothing collecting: 0.000
~ seconds the three threads lost with those passes running: 0.127
~ lost per thread per pass, in milliseconds: 8

    Those threads never touched the heap the collector was walking and never called
    anything in the gc module. They were stopped anyway, because the collector needs
    every reference count in the process to hold still while it works out which ones
    are only kept alive by the cycle it is looking at.

    The stopping uses the same machinery M07 described. A thread that is running Python
    gets a bit set on its eval breaker and parks itself between two bytecode
    instructions. A thread that is already blocked in C, waiting on a socket or a lock,
    is marked parked without being woken at all, which is why a program full of threads
    waiting on IO costs the collector nothing to stop.
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

The two totals line up almost perfectly. Three threads, five passes, and the time they lost between them is three times what the collector spent, which is the arithmetic you get when all three are stopped for all of it. None of those threads shared a single object with the heap being walked.

## Why it can afford to walk everything

A build with no generations walks the whole heap on every pass, and the pause above is what that costs. So there has to be something making it cheaper than it sounds, and there is.

Before the real pass starts, the collector does a quick sweep from a known root, follows every reference it can reach, and sets the alive bit on everything it lands on [Python/gc_free_threading.c:1376-1401@v3.15.0rc1#gc_mark_alive_from_roots](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L1376-L1401). The pass proper then skips anything wearing that bit. In most programs that is nearly everything, because nearly everything really is reachable from `sys.modules`.

![the mark alive sweep setting a bit that lets the real pass skip most of the heap](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m08-stop-the-world/diagrams/mark-alive-first.svg)

The [mark alive pass](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#mark-alive-pass) is the reason this build's collector is not ruinous, and it is also why `gc.freeze()` turns it off. A frozen object is skipped anyway, so marking it alive is wasted work, and worse, writing the bit defeats the whole point of freezing before a fork [Python/gc_free_threading.c:2099-2116@v3.15.0rc1#freeze_active](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2099-L2116).

Freezing itself is different here too. On the ordinary build it moves objects between lists. Here there are no lists, so it walks the heap once and sets a bit on each object [Python/gc_free_threading.c:2453-2462@v3.15.0rc1#visit_freeze](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L2453-L2462).

On 3.15 you can see roughly how much a pass considered, because `gc.get_stats()` gained a candidates field.

gc.get_stats reports a candidates count on 3.15, and after a full pass over a large heap that count is at least as large as the number of objects the collector is tracking

In [ ]:
gc.collect()
tracked = len(gc.get_objects())
stats = gc.get_stats()[2]

if "candidates" in stats:
    print(f"  objects the collector is tracking     {tracked}")
    print(f"  candidates all full passes have seen  {stats['candidates']}")
    print(f"  full passes so far                    {stats['collections']}")
    print(f"  seconds spent in them                 {stats['duration']:.4f}")
else:
    print("  this version does not report candidates, so there is nothing to compare")
    print(f"  objects the collector is tracking      {tracked}")

> **Version note.** On 3.14 there is no candidates field and the cell prints the short version. The counts themselves are whatever your session has done, so they will not match anybody else's.

## What deferred counting does to the collector

M06 introduced deferred reference counting: for functions, classes and modules, the interpreter stops counting references taken from the evaluation stack, so the count on those objects is wrong on purpose.

That is a problem for a collector that works by comparing counts. Its fix is to walk every thread's stack and add one for each deferred reference it finds there, which puts the count back to what it should be for the duration of the pass [Python/gc_free_threading.c:445-478@v3.15.0rc1#gc_visit_thread_stacks](https://github.com/python/cpython/blob/v3.15.0rc1/Python/gc_free_threading.c#L445-L478).

There is a corner where it cannot do that. A thread caught in the middle of closing a stack reference has a frame with no valid stack pointer, and the collector cannot read that frame safely. When it sees one, it gives up on collecting any object with deferred counting at all for that pass and treats them as reachable. Nothing leaks; the pass just does less.

This is the sort of thing that only shows up when you go looking. It is also why the free threaded collector is a good deal more code than the ordinary one for the same job.

## Try it yourself

Three things.

Take the timing cell and change 200000 to 20000, then to 2000000 if your machine has the memory. The pass time should track the number of cycles almost linearly, which is what walking everything means. Then run it once with `gc.freeze()` called just before, and see how much of the cost was walking objects that were never going to be freed.

Pull the image and run the recordings yourself: `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded python3 -` and paste in any of the three programs. The counter one is the most fun to change, because you can add a second helper thread and watch the reader fall behind by 1024 instead of 512.

Take the stop the world program and give the spinning threads something to do that blocks, like `time.sleep(0.001)` in the loop. A sleeping thread is parked without being woken, so the time it loses should drop sharply even though the collector is doing exactly the same work.

## What you now know

The free threaded build does not keep the collector's three lists. `struct _gc_runtime_state` has the thresholds and the counts and nothing to hang objects off.

The collector finds objects by walking the memory allocator's heaps instead. What used to be a place in a linked list is now a bit in `ob_gc_bits`, the byte M06 found at offset 11.

So every pass walks everything. `gc.get_objects(generation=N)` returns the whole heap for all three values of N, and `gc.collect(0)` frees a cycle no matter how many passes it has survived. The generation argument is accepted and ignored.

The counter is approximate. Each thread buffers 512 allocations before touching the shared count, so a thread reading it sees another thread's work up to 512 objects late. Reading it from the thread that did the allocating is exact, because that flushes first.

A collection stops every other thread. Threads running Python park between bytecode instructions on an eval breaker bit; threads blocked in C are marked parked without being woken. They stay stopped for the whole search, and only start again before finalizers run.

Three spinning threads lose about three times what the collector spends, which is what being stopped for all of it looks like.

The thing that makes walking everything affordable is a mark alive sweep from a known root, which sets a bit on everything obviously reachable so the real pass can skip it. `gc.freeze()` turns that sweep off, because a frozen object would be skipped anyway and writing the bit is exactly what freezing exists to avoid.

Deferred reference counting makes the counts wrong on purpose, so the collector walks every thread's stack and adds them back for the duration of the pass.

## What is next

M09 is the last of the memory lessons, and it is the practical one. Everything so far has been about how CPython decides what to free. That lesson is about what to do when it decides not to free something you expected it to, which happens more often than the machinery would suggest and almost never for the reason you first guess.